In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)

text = open(path_to_file, 'rb').read().decode(encoding='utf-8')

print("Dataset length:", len(text))
print(text)

In [ ]:
tokenizer = Tokenizer()

tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

print("Vocabulary size:", total_words)

Vocabulary size: 12633


In [ ]:
input_sequences = []

for line in text.split('\n'):

    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):

        n_gram_sequence = token_list[:i+1]

        input_sequences.append(n_gram_sequence)

print("Total sequences:", len(input_sequences))

Total sequences: 171312


In [ ]:
max_sequence_len = max([len(x) for x in input_sequences])

input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_sequence_len,
    padding='pre'
)


max_sequence_len

16

In [ ]:
X = input_sequences[:,:-1]

y = input_sequences[:,-1]



In [ ]:
model = Sequential()

model.add(Embedding(total_words, 100, input_shape=(max_sequence_len,)))

model.add(LSTM(128))

model.add(Dense(total_words, activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 16, 100)        │     1,263,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 12633)          │     1,629,657 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,010,205 (11.48 MB)

 Trainable params: 3,010,205 (11.48 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:


history = model.fit(
    X,
    y,
    epochs=10,
    batch_size=64
)

Epoch 1/10
2677/2677 ━━━━━━━━━━━━━━━━━━━━ 237s 88ms/step - accuracy: 0.0420 - loss: 6.8035
Epoch 2/10
2677/2677 ━━━━━━━━━━━━━━━━━━━━ 232s 87ms/step - accuracy: 0.0843 - loss: 6.1682
Epoch 3/10
2677/2677 ━━━━━━━━━━━━━━━━━━━━ 236s 88ms/step - accuracy: 0.1000 - loss: 5.8278
Epoch 4/10
2677/2677 ━━━━━━━━━━━━━━━━━━━━ 237s 88ms/step - accuracy: 0.1114 - loss: 5.5509
Epoch 5/10
2677/2677 ━━━━━━━━━━━━━━━━━━━━ 240s 90ms/step - accuracy: 0.1217 - loss: 5.2985
Epoch 6/10
2677/2677 ━━━━━━━━━━━━━━━━━━━━ 264s 90ms/step - accuracy: 0.1313 - loss: 5.0670
Epoch 7/10
2677/2677 ━━━━━━━━━━━━━━━━━━━━ 262s 90ms/step - accuracy: 0.1430 - loss: 4.8542
Epoch 8/10
2677/2677 ━━━━━━━━━━━━━━━━━━━━ 242s 90ms/step - accuracy: 0.1573 - loss: 4.6527
Epoch 9/10
2677/2677 ━━━━━━━━━━━━━━━━━━━━ 242s 90ms/step - accuracy: 0.1751 - loss: 4.4638
Epoch 10/10
2677/2677 ━━━━━━━━━━━━━━━━━━━━ 261s 90ms/step - accuracy: 0.1957 - loss: 4.2854


In [ ]:
seed_text = input("Type something \t")

next_words = 20

for _ in range(next_words):

    token_list = tokenizer.texts_to_sequences([seed_text])[0]

    token_list = pad_sequences(
        [token_list],
        maxlen=max_sequence_len-1,
        padding='pre'
    )

    predicted = np.argmax(model.predict(token_list), axis=-1)

    for word, index in tokenizer.word_index.items():

        if index == predicted:

            seed_text += " " + word
            break

print(seed_text)

Type something 	 words
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
 words i am to the king of the king of the king of the king of tides yonder's yonder's yonder's the
